## AI Mystery Detective Game 

In [2]:
#
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
import uuid

## Step 1: Generate the case

In [3]:
MODEL = "llama3.2:1b"

Ollama_Base_Url = 'http://localhost:11434/v1';

ollama = OpenAI(base_url=Ollama_Base_Url, api_key="ollama_api_key");


In [4]:
def call_llm(messages, model=MODEL): 
    response = ollama.chat.completions.create(
        model = model, 
        messages=messages, 
        response_format={'type': 'json_object'}, 
        temperature=0,
    )

    return response.choices[0].message.content;

In [5]:
case_system_prompt = """You generate a detective mystery case as JSON.

Output ONLY the JSON object. No markdown, no ```json fences, no text before or after.

Use EXACTLY this structure and fill every field:
{
  "title": "short case name",
  "description": "2-3 sentences describing the crime",
  "suspects": [
    {"name": "name", "motive": "why they might do it", "secret": "a hidden secret"},
    {"name": "name", "motive": "why they might do it", "secret": "a hidden secret"},
    {"name": "name", "motive": "why they might do it", "secret": "a hidden secret"}
  ],
  "clues": [
    "clue 1",
    "clue 2",
    "clue 3",
    "clue 4 (this one is misleading)",
    "clue 5"
  ],
  "solution": {"culprit": "name of one suspect", "reason": "why they are guilty"}
}

Rules:
- Exactly 3 suspects and 5 clues.
- One clue must be misleading.
- "culprit" must match one suspect name.
"""

In [6]:
caes_user_prompt = """
Generate ONE new mystery case set in: a locked mansion during a dinner party.
Follow the exact JSON structure. Output only the JSON object
"""

In [7]:
#Validate Case
def validate_case(case):

    if not isinstance(case, dict):
        raise ValueError("case must be a JSON object (dict)")

    # 1. Required top-level keys
    for key in ("title", "description", "suspects", "clues", "solution"):
        if key not in case:
            raise ValueError(f"missing key: {key}")

    # 2. Suspects: exactly 3, each with the right fields
    suspects = case["suspects"]
    if not isinstance(suspects, list) or len(suspects) != 3:
        raise ValueError("need exactly 3 suspects")
    for i, s in enumerate(suspects):
        for field in ("name", "motive", "secret"):
            if field not in s:
                raise ValueError(f"suspect {i} missing field: {field}")

    # 3. Clues: exactly 5
    clues = case["clues"]
    if not isinstance(clues, list) or len(clues) != 5:
        raise ValueError("need exactly 5 clues")

    # 4. Solution: has culprit + reason, and culprit matches a real suspect
    solution = case["solution"]
    for field in ("culprit", "reason"):
        if field not in solution:
            raise ValueError(f"solution missing field: {field}")
    suspect_names = [s["name"] for s in suspects]
    if solution["culprit"] not in suspect_names:
        raise ValueError(f"culprit '{solution['culprit']}' is not one of {suspect_names}")

    return case



In [ ]:
def generate_case(): 
    response = call_llm(messages=[
        {"role": "system", "content": case_system_prompt},
        {"role": "user", "content": caes_user_prompt},
    ])


    # Ensure the case format from LLM is correct
    case_file = validate_case(json.loads(response));

    
    for key, value in case_file.items(): 
        print(f"{key} : {value}")

    return case_file
    
entire_case_file = generate_case()


title : Mansion of Deception
description : A wealthy businessman hosts a dinner party at his luxurious mansion, but the evening takes a deadly turn when one of the guests is found murdered.
suspects : [{'name': 'Richard Langley', 'motive': 'Economic gain', 'secret': 'He stood to lose a large sum of money if the business was sold to a rival company'}, {'name': 'Sophia Patel', 'motive': 'Revenge', 'secret': 'She had a long-standing grudge against the businessman'}, {'name': 'Dr. Henry Lee', 'motive': 'Medical research', 'secret': "He had been working on a top-secret project that could have ruined Langley's business if it fell into the wrong hands"}]
clues : ["A torn piece of fabric was found near the victim's body with a faint scent of lavender", "A security camera caught a glimpse of a figure in a black coat lurking around the mansion's garden", "The victim's phone was found with a cryptic message on the screen: 'You'll never find me'", 'A guest reported seeing Sophia Patel arguing with

In [ ]:
# Game state will act as the entire context of the game progress
def initialize_game_state():
    return {
        "actions": [],           # every action the player has taken 
        "discovered_clues": [],  # clues revealed so far
        "conversations": {},     # suspect name -> chat history
    }

game_state = initialize_game_state()
print(game_state)

{'actions': [], 'discovered_clues': [], 'conversations': {}}


In [10]:
action_system_prompt = """You are a detective assistant. Suggest the player's next investigation steps.

You are given the case description, the suspects, and what the player has already done.
You do NOT know who is guilty. Never name or hint at a culprit.

Output ONLY this EXACT JSON structure. No markdown, no text before or after:
{
  "recommendations": [
    {"type": "Conversation", "action": "interrogate <suspect name>", "why": "1 short sentence"},
    {"type": "Auto", "action": "review timeline", "why": "1 short sentence"}
  ]
}

"type" must be EXACTLY one of these three values:
- "Conversation": the action is talking to a suspect (needs an LLM chat).
- "Describe": the game describes a scene and the player reacts to what they found.
- "Auto": the game simply reveals the result (no conversation).

Rules:
- Give 2 or 3 recommendations.
- "action" must use ONLY suspect names and places from the input.
- Prefer actions the player has NOT done yet.
- Do NOT invent suspects that are not in the input.
"""


def build_action_prompt(case_file, game_state):
    """Build a spoiler-free menu from the CURRENT state.
    Passes the player-visible description + suspect names only --
    never case_file['clues'] or case_file['solution']."""
    menu = {
        "description": case_file["description"],
        "suspects": [s["name"] for s in case_file["suspects"]],
        "already_done": game_state["actions"],
    }
    return f"""Here is the current investigation. Recommend what the player
should do next, using ONLY the suspects and places in this data:

{json.dumps(menu, indent=2)}

Follow the exact JSON structure. Output only the JSON object.
"""


In [11]:
ALLOWED_TYPES = ("Conversation", "Describe", "Auto")

def generate_action(case_file, game_state):
    response = call_llm(messages=[
        {"role": "system", "content": action_system_prompt},
        {"role": "user", "content": build_action_prompt(case_file, game_state)},
    ])
    print(response)

    actions = json.loads(response)
    # Validate the json format
    if not isinstance(actions, dict):
        raise ValueError("response must be a JSON object (dict)")

    if "recommendations" not in actions or not isinstance(actions["recommendations"], list):
        raise ValueError("Missing or invalid 'recommendations' key at the root")

    for index, rec in enumerate(actions["recommendations"]):
        for key in ("type", "action", "why"):
            if key not in rec:
                raise ValueError(f"Missing key '{key}' in recommendation at index {index}")
        if rec["type"] not in ALLOWED_TYPES:
            raise ValueError(f"recommendation {index} has invalid type '{rec['type']}', must be one of {ALLOWED_TYPES}")

    return actions


actions = generate_action(entire_case_file, game_state)


{
  "recommendations": [
    {"type": "Describe", "action": "show Richard Langley's mansion", "why": "1 short sentence"},
    {"type": "Describe", "action": "show Sophia Patel's alibi", "why": "1 short sentence"},
    {"type": "Describe", "action": "show Dr. Henry Lee's alibi", "why": "1 short sentence"}
  ]
}


## Investigation Loop

In [12]:
def investigate_action() : 
    pass;

In [13]:
current_chat = [];

In [ ]:
INTERROGATION_FORMAT_INSTRUCTIONS = """
Return ONLY this JSON object. No markdown, no text before or after:
{"answer": "your answer, in character, 1-3 sentences"}
"""


def find_suspect(case_file, suspect_name):
    for suspect in case_file["suspects"]:
        if suspect["name"] == suspect_name:
            return suspect
    raise ValueError(f"'{suspect_name}' is not a suspect in this case")


def build_interrogation_system_prompt(case_file, suspect):
    
    return f"""You are {suspect['name']}, a suspect being interrogated by a detective
investigating this case: {case_file['title']}

Case background (public knowledge): {case_file['description']}

Your motive (known only to you): {suspect['motive']}
Your secret (known only to you): {suspect['secret']}

Stay in character as {suspect['name']}. You may be evasive, defensive, or
even deceptive about your secret, but never break character, never mention
you are an AI, and never reveal who actually committed the crime -- you
don't know the solution.

{INTERROGATION_FORMAT_INSTRUCTIONS}
"""


def interrogate_suspect(case_file, game_state, suspect_name, question, current_chat):
    suspect = find_suspect(case_file, suspect_name)
    system_prompt = build_interrogation_system_prompt(case_file, suspect)

    # Keep the current chat history
    current_chat.append({"role": "user", "content": question})

    response = call_llm([
        {"role": "system", "content": system_prompt},
        *current_chat,
    ])
    print(response)

    parsed = json.loads(response)
    # Validate Json Format
    if not isinstance(parsed, dict) or not isinstance(parsed.get("answer"), str):
        raise ValueError(f"interrogation response missing string 'answer' key: {response}")

    # keep the raw JSON turn in history so the model's own prior outputs stay
    # consistent with the "always JSON" instruction on later questions
    current_chat.append({"role": "assistant", "content": response})

    game_state["conversations"][suspect_name] = current_chat
    game_state["actions"].append(f"interrogate {suspect_name}: {question}")

    return parsed["answer"]


answer = interrogate_suspect(
    case_file=entire_case_file,
    game_state=game_state,
    suspect_name="Richard Langley",
    question="///",
    current_chat=current_chat,
)
print(answer)
current_chat


{"answer": "I'll do whatever it takes to protect my interests, but I'm not going down without a fight. I've invested too much time and money into this business to let it fall apart over a single murder. I'll play the game, but I'll make sure the truth comes out, no matter what it takes."}
I'll do whatever it takes to protect my interests, but I'm not going down without a fight. I've invested too much time and money into this business to let it fall apart over a single murder. I'll play the game, but I'll make sure the truth comes out, no matter what it takes.


[{'role': 'user', 'content': 'just accept the fate and go to jail man'},
 {'role': 'assistant',
  'content': '{"answer": "I\'ll do whatever it takes to protect my interests, but I\'m not going down without a fight. I\'ve invested too much time and money into this business to let it fall apart over a single murder. I\'ll play the game, but I\'ll make sure the truth comes out, no matter what it takes."}'}]

In [15]:
def build_describe_system_prompt(case_file, game_state):
    """Spoiler-safe scene narrator. Knows the case description and the full
    clue list (so it can ground a scene in a real clue), and what's already
    been discovered (so it doesn't repeat itself) -- but never the solution."""
    known_clues = case_file["clues"]
    already_found = game_state["discovered_clues"]

    return f"""You are the narrator of a detective mystery game: {case_file['title']}

Case background: {case_file['description']}

Clues that exist in this case (only draw from these -- never invent new ones):
{json.dumps(known_clues, indent=2)}

Clues the player has already discovered (do not repeat these):
{json.dumps(already_found, indent=2)}

The detective is examining a specific person or place. Write a short,
atmospheric in-game description of what they find.

Rules:
- If the scene plausibly surfaces one of the UN-discovered clues above, reveal
  it through the narration and set "revealed_clue" to that EXACT clue string.
- If nothing new applies (e.g. the player is re-examining something, or this
  scene has no clue tied to it), write flavor description only and set
  "revealed_clue" to null.
- NEVER mention or hint at who the culprit is, or anything from the solution.
- Do not break character or mention you are an AI.

Return ONLY this JSON object. No markdown, no text before or after:
{{
  "narration": "2-4 sentences of in-game scene description",
  "revealed_clue": "exact clue text, or null"
}}
"""

def build_describe_user_prompt(action):
    return f"""The detective chooses to: {action}

Describe what they find. Follow the exact JSON structure. Output only the JSON object.
"""

def generate_description(action, game_state, case_file): 
    response = call_llm([
        {"role": "system", "content": build_describe_system_prompt(case_file, game_state)},
        {"role": "user", "content": build_describe_user_prompt(action)}
    ])

    return json.loads(response);


#Get description
description =  generate_description("show Richard Langley's mansion", game_state, entire_case_file);

In [23]:
def build_player_describe_system_prompt(case_file, game_state,): 
    known_clues = case_file["clues"]
    already_found = game_state["discovered_clues"]

    return f"""You are the game master of a detective mystery game: {case_file['title']}

Case background: {case_file['description']}

Clues that actually exist in this case (the ONLY clues you may recognize):
{json.dumps(known_clues, indent=2)}

Clues the player has already discovered (do not report these again):
{json.dumps(already_found, indent=2)}

The player will describe, in their own words, what they personally notice or
deduce about a scene you just showed them. Judge whether their observation
genuinely demonstrates they noticed one of the real clues above -- not just
that they sound confident or are asking a question.

Rules:
- Only include a clue in "found_clues" if the player's own words genuinely
  correspond to that clue's content (paraphrasing is fine, guessing is not).
- Never report a clue that is already in the "already discovered" list.
- Never invent a clue that is not in the clue list above.
- NEVER reveal or hint at the solution/culprit, even if the player guesses it.
- If the player's observation doesn't match any real clue, "found_clues" must
  be an empty array -- do not reward them just for writing something.

Return ONLY this JSON object. No markdown, no text before or after:
{{
  "feedback": "1-2 sentences, in-character reaction to what the player noticed",
  "found_clues": ["exact clue text from the list above", "..."]
}}
"""

player_describe_user_prompt = """
Follow the exact JSON structure. Output only the JSON object. No markdown, no text before or after.
"""

#allow player to describe what they notice with couple sentences
def describe(player_text, description, game_state, case_file):
  response = call_llm(messages=[
        {"role": "system", "content": build_player_describe_system_prompt(case_file, game_state)},
        {"role": "user", "content": f"""
{player_describe_user_prompt}

Here is the narration of the actual scence
{json.dumps(description, indent=2)}

Here is how the player thoughts on the description. 
{player_text}
        """},
    ])
  return json.loads(response);

describe(player_text="...", description=description, game_state=game_state, case_file=entire_case_file)


{'feedback': "The detective steps out of the car and onto the manicured lawn of Richard Langley's mansion, the warm summer air filled with the sweet scent of blooming flowers. The mansion's facade is a stunning example of Gothic Revival architecture, with towering spires and turrets that seem to reach for the sky. A winding path leads to the entrance, flanked by two massive stone statues of gargoyles. As the detective approaches the door, they notice a faint scent of lavender wafting from within.",
 'found_clues': ["a torn piece of fabric was found near the victim's body with a faint scent of lavender"]}

In [17]:
def extract_suspect_name(action, case_file):
    for suspect in case_file["suspects"]:
        if suspect["name"].lower() in action.lower():
            return suspect["name"]
    raise ValueError(f"no suspect name found in action: '{action}'")


def format_timeline(game_state):
    if not game_state["actions"]:
        return "You haven't investigated anything yet."
    steps = "\n".join(f"{i + 1}. {a}" for i, a in enumerate(game_state["actions"]))
    return f"Timeline of your investigation:\n{steps}"


def format_suspect_list(case_file):
    # names only -- motive/secret are each suspect's own private knowledge,
    # same rule build_interrogation_system_prompt and build_action_prompt follow
    names = "\n".join(f"- {s['name']}" for s in case_file["suspects"])
    return f"Suspects in this case:\n{names}"


def format_clue_summary(case_file, game_state):
    found = game_state["discovered_clues"]
    if not found:
        return f"{case_file['title']}: no clues discovered yet."
    listed = "\n".join(f"- {c}" for c in found)
    return f"Clues discovered so far:\n{listed}"


def reveal_known_info(action, game_state, case_file):
    """The 'Auto' handler: just reveal information the game already has --
    nothing new to discover, so this never calls the LLM (zero API cost)."""
    game_state["actions"].append(action)
    lowered = action.lower()

    if "timeline" in lowered or "review" in lowered:
        return format_timeline(game_state)
    if "suspect" in lowered:
        return format_suspect_list(case_file)
    return format_clue_summary(case_file, game_state)


def auto_generate_content(recommendation, game_state, case_file, player_input=None):
    """Dispatch ONE recommendation from generate_action()'s 'recommendations'
    list to the right handler, based on its 'type':
      - "Conversation" -> interrogate_suspect (needs a suspect + a question)
      - "Describe"     -> generate_description (narrates a scene, may reveal a clue)
      - "Auto"         -> reveal_known_info (no LLM call, just shows existing data)
    """
    action_type = recommendation["type"]
    action = recommendation["action"]

    if action_type == "Conversation":
        suspect_name = extract_suspect_name(action, case_file)
        chat_history = game_state["conversations"].get(suspect_name, [])
        question = player_input or "Where were you last night?"
        content = interrogate_suspect(case_file, game_state, suspect_name, question, chat_history)

    elif action_type == "Describe":
        result = generate_description(action, game_state, case_file)
        revealed = result.get("revealed_clue")
        if revealed and revealed not in game_state["discovered_clues"]:
            game_state["discovered_clues"].append(revealed)
        game_state["actions"].append(action)
        content = result["narration"]

    elif action_type == "Auto":
        content = reveal_known_info(action, game_state, case_file)

    else:
        raise ValueError(f"Unknown action type: '{action_type}'")

    print(content)
    return content


# try it against the actual recommendations generate_action() produced
auto_generate_content({"type": "Auto", "action": "review Dr. Henry Lee's alibi", "why": "1 short sentence"}, game_state, entire_case_file)


Timeline of your investigation:
1. interrogate Richard Langley: just accept the fate and go to jail man
2. review Dr. Henry Lee's alibi


"Timeline of your investigation:\n1. interrogate Richard Langley: just accept the fate and go to jail man\n2. review Dr. Henry Lee's alibi"

## Solving Case

In [18]:
# Reasoning must score at least this (0-10) for a correct accusation to count as "solved".
# A 1B model grades loosely, so tune this after seeing real scores.
PASSING_REASONING_SCORE = 6


def build_solve_system_prompt(case_file, game_state, accused_name, culprit_matched):
    """Judge prompt. This is the ONLY prompt that sees the hidden solution.
    The model grades the player's REASONING only -- Python has already decided
    whether the accused suspect is right (culprit_matched) and tells the model
    as a fixed fact it must not reconsider."""
    solution = case_file["solution"]
    already_found = game_state["discovered_clues"]

    return f"""You are the judge of a detective mystery game: {case_file['title']}

Case background: {case_file['description']}

The hidden solution (never quote it back to the player):
- Culprit: {solution['culprit']}
- Reason: {solution['reason']}

Clues the player has actually discovered (the ONLY evidence they may be credited for):
{json.dumps(already_found, indent=2)}

The player accused: {accused_name}
Is that the correct culprit? {"YES" if culprit_matched else "NO"}  (this is already decided -- do NOT reconsider it)

Grade ONLY the quality of the player's reasoning, from 0 to 10:
- 0-3: no real reasoning, guessing, or reasoning that ignores the evidence.
- 4-6: partly grounded in discovered clues but with gaps.
- 7-10: clearly connects discovered clues to the accused's guilt.

Rules:
- Do NOT raise the score because the player sounds confident or argues forcefully.
- Only list a clue in "decisive_evidence" if it is in the discovered list above
  AND the player's reasoning actually relies on it. Copy it EXACTLY.
- Do not credit evidence the player has not discovered, even if it is true.

Return ONLY this JSON object. No markdown, no text before or after:
{{
  "reasoning_score": 0,
  "decisive_evidence": ["exact discovered clue text", "..."],
  "explanation": "1-2 sentences on the strengths or gaps in the player's reasoning"
}}
"""


def build_solve_user_prompt(user_description_on_the_case):
    return f"""The player's reasoning:
{user_description_on_the_case}

Grade it. Follow the exact JSON structure. Output only the JSON object.
"""


def validate_judgement(judgement, game_state):
    """Validate the judge's JSON, then let PYTHON clean it up: clamp nothing
    silently -- reject bad shapes, but drop any 'evidence' the player never found."""
    if not isinstance(judgement, dict):
        raise ValueError("judgement must be a JSON object (dict)")

    score = judgement.get("reasoning_score")
    # bool is a subclass of int in Python, so exclude it explicitly
    if isinstance(score, bool) or not isinstance(score, int) or not 0 <= score <= 10:
        raise ValueError(f"reasoning_score must be an int from 0 to 10, got: {score!r}")

    evidence = judgement.get("decisive_evidence")
    if not isinstance(evidence, list) or not all(isinstance(e, str) for e in evidence):
        raise ValueError("decisive_evidence must be a list of strings")

    if not isinstance(judgement.get("explanation"), str):
        raise ValueError("explanation must be a string")

    # Python enforcement: only discovered clues can be credited
    judgement["decisive_evidence"] = [
        e for e in evidence if e in game_state["discovered_clues"]
    ]
    return judgement


In [ ]:
VERDICT_MESSAGES = {
    "incorrect": "That's not the culprit. Keep investigating and try again.",
    "investigate_more": "You may be onto something, but your reasoning isn't convincing yet. Investigate more.",
    "solved": "Case closed! You identified the culprit and backed it up.",
}


def decide_verdict(culprit_matched, reasoning_score):
    """YOUR PART: this is where Python enforces the final result.

    Inputs:
      culprit_matched  -- bool, already computed by Python (NOT the model)
      reasoning_score  -- int 0-10 from the judge, validated
      PASSING_REASONING_SCORE -- threshold constant (cell above)

    Return exactly one of: "incorrect", "investigate_more", "solved"
      - wrong culprit                          -> "incorrect"
      - right culprit, score below threshold   -> "investigate_more"
      - right culprit, score at/above it       -> "solved"

    Things to consider: is "at the threshold" a pass or a fail? Should the
    wrong-culprit check come first so the score can never rescue a wrong answer?
    """
    # TODO: write 5-8 lines
    raise NotImplementedError("write decide_verdict")


def solveTheCase(user_description_on_the_case, case_file, game_state, accused_name):
    """accused_name is passed explicitly (not parsed out of the free text) so a
    description like "not Sophia, it was Richard" can't be misread."""
    find_suspect(case_file, accused_name)  # raises if not a real suspect

    # 1. PYTHON decides the answer -- the model never gets a vote on this.
    culprit_matched = accused_name == case_file["solution"]["culprit"]

    # 2. ONE LLM call grades the reasoning only.
    response = call_llm([
        {"role": "system", "content": build_solve_system_prompt(case_file, game_state, accused_name, culprit_matched)},
        {"role": "user", "content": build_solve_user_prompt(user_description_on_the_case)},
    ])
    judgement = validate_judgement(json.loads(response), game_state)

    # 3. PYTHON turns (culprit, score) into the final verdict.
    verdict = decide_verdict(culprit_matched, judgement["reasoning_score"])

    result = {
        "verdict": verdict,
        "message": VERDICT_MESSAGES[verdict],
        "reasoning_score": judgement["reasoning_score"],
    }
   
    # accusation it could leak details of the real solution.
    if culprit_matched:
        result["decisive_evidence"] = judgement["decisive_evidence"]
        result["explanation"] = judgement["explanation"]

    game_state["actions"].append(f"accuse {accused_name}: {verdict}")
    return result
